In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

In [18]:
# 1. Baca dataset
data = pd.read_csv("Survei Penggunaan Platform Belajar Online pada Mahasiswa (Jawaban) - Form Responses 1.csv")

# 2. Siapkan kolom "teks" (gabungan 2 fitur) dan "label" sesuai dataset
data["teks"] = data["Tujuan utama Anda menggunakan platform belajar online?"] + " " + data["Jenis materi yang paling Anda sukai saat belajar online?"]
data["label"] = data["Menurut Anda, berapa persentase materi/kursus/modul yang biasanya Anda selesaikan?"]

print("Data shape:", data.shape)
print("\nFirst few rows:")
print(data.head())

Data shape: (103, 16)

First few rows:
                      Cap waktu            Nama lengkap  \
0  2026/04/08 10:24:53 PM GMT+7         Fizar Erlansyah   
1  2026/04/08 10:27:49 PM GMT+7  Muhammad Rayhan Mumtaz   
2  2026/04/08 10:28:25 PM GMT+7      Panji Kurnia Akbar   
3  2026/04/08 10:31:24 PM GMT+7       Azkiyah zhafirah    
4  2026/04/08 10:32:41 PM GMT+7            zavira verga   

  Asal universitas (cantumkan nama lengkap) Jurusan / program studi  Angkatan  \
0              Universitas Pembangunan Jaya        Sistem Informasi      2024   
1              Universitas Pembangunan Jaya        Sistem Informasi      2024   
2              Universitas Pembangunan Jaya        Sistem Informasi      2024   
3                    Universitas Pancasila                 Farmasi       2025   
4              Universitas Pembangunan Jaya             accounting       2025   

    Usia Anda? Status/aktivitas utama Anda saat ini?  \
0  18–24 tahun                     Mahasiswa/pelajar   
1  18–2

In [19]:
# 3. Siapkan fitur (X) dan label (y)
X = data["teks"]
y = data["label"]

print("Jumlah data:", len(X))
print("Label yang ada:", y.unique())

Jumlah data: 103
Label yang ada: <StringArray>
['81–100%', '41–60%', '61–80%', '21–40%', '0–20%']
Length: 5, dtype: str


In [20]:
# 4. Preprocessing sederhana
X = X.astype(str).str.lower()
X = X.str.replace(r'[^\w\s]+', '', regex=True)  # Hapus tanda baca

print("Preprocessing selesai")
print("Contoh teks setelah preprocessing:")
print(X.iloc[0])

Preprocessing selesai
Contoh teks setelah preprocessing:
pengembangan karier campuran keduanya


In [21]:
# 5. Ubah teks ke angka (Vectorization)
X_text_original = X.copy()  # Simpan teks asli untuk referensi
vectorizer = CountVectorizer()
X_vector = vectorizer.fit_transform(X)

print("Jumlah fitur:", X_vector.shape[1])
print("Shape X_vector:", X_vector.shape)

Jumlah fitur: 15
Shape X_vector: (103, 15)


In [22]:
# 6. Split data ke training dan testing
X_train, X_test, y_train, y_test, X_train_text, X_test_text = train_test_split(
    X_vector, y, X_text_original, test_size=0.2, random_state=42
)

print("Jumlah training data:", X_train.shape[0])
print("Jumlah testing data:", X_test.shape[0])

Jumlah training data: 82
Jumlah testing data: 21


In [23]:
# 7. Train model Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model berhasil dilatih")

Model berhasil dilatih


In [24]:
# 8. Prediksi pada data baru (sesuai format dataset)
from IPython.display import display, HTML

print("\n" + "="*120)
print("PREDIKSI PADA DATA BARU".center(120))
print("="*120)

# Data baru dari mahasiswa
data_baru = pd.DataFrame({
    'Tujuan': ['Keperluan akademik', 'Upgrade skill', 'Hobi / minat pribadi', 'Pengembangan karier'],
    'Jenis Materi': ['Video', 'Campuran keduanya', 'Teks / artikel / modul', 'Video']
})

# Gabung kolom seperti training
data_baru['teks'] = data_baru['Tujuan'] + " " + data_baru['Jenis Materi']

# Preprocessing
data_baru['teks'] = data_baru['teks'].astype(str).str.lower()
data_baru['teks'] = data_baru['teks'].str.replace(r'[^\w\s]+', '', regex=True)

# Vectorize
X_baru = vectorizer.transform(data_baru['teks'])

# Prediksi
prediksi_baru = model.predict(X_baru)

# Tampilkan hasil dengan style tabel
hasil_prediksi = pd.DataFrame({
    'Tujuan': data_baru['Tujuan'].values,
    'Jenis Materi': data_baru['Jenis Materi'].values,
    'Prediksi Persentase': prediksi_baru
})

style = (
    hasil_prediksi.style
    .set_table_styles([
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('background-color', '#ffffff'), ('color', '#111111')]},
        {'selector': 'th', 'props': [('background-color', '#e6e6e6'), ('color', '#111111'), ('border', '1px solid #333'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('background-color', '#ffffff'), ('color', '#111111'), ('border', '1px solid #333'), ('padding', '8px')]},
        {'selector': 'tr:nth-child(even) td', 'props': [('background-color', '#f7f7f7')]} 
    ])
    .hide(axis='index')
)

print("\nTabel hasil prediksi:")
display(HTML(style.to_html()))
print("="*120)


                                                PREDIKSI PADA DATA BARU                                                 

Tabel hasil prediksi:


Tujuan,Jenis Materi,Prediksi Persentase
Keperluan akademik,Video,81–100%
Upgrade skill,Campuran keduanya,21–40%
Hobi / minat pribadi,Teks / artikel / modul,0–20%
Pengembangan karier,Video,41–60%
